In [2]:
import numpy as np
import ot_utils
import comparison_utils
from tqdm import tqdm
import math
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from sklearn.metrics import pairwise_distances


n_classes = 8
labels = np.repeat(np.arange(8), 1000)

w2_block_files = []
sw2_block_files = []

for i in range(n_classes):
    for j in range(i, n_classes):
        w2_block_files.append(f"../../mini_w2/block_{i}_{j}.npy")

for i in range(n_classes):
    for j in range(i, n_classes):
        sw2_block_files.append(f"../../mini_sw/block_{i}_{j}_sw2.npy")

w2_matrix = ot_utils.combine_distance_blocks(w2_block_files)
sw2_matrix = ot_utils.combine_distance_blocks(sw2_block_files)

In [3]:
w2_rel_rep, sw2_rel_rep = comparison_utils.relative_rep(
    w2_matrix, sw2_matrix, labels, n_classes, anchors_per_class=10, seed=1
)

del w2_matrix, sw2_matrix

In [ ]:
pairwise_eud_dist_w2 = pairwise_distances(
    w2_rel_rep,
    w2_rel_rep,
    metric=lambda u, v: comparison_utils.eud_dist(u, v),
    n_jobs=-1,  # use all CPUs to parallelize
)

pairwise_eud_dist_sw2 = pairwise_distances(
    sw2_rel_rep,
    sw2_rel_rep,
    metric=lambda u, v: comparison_utils.eud_dist(u, v),
    n_jobs=-1,  # use all CPUs to parallelize
)

In [5]:
first_class = 4
second_class = 5

idx_list_1 = list(range(first_class * 1000, (first_class + 1) * 1000))
idx_list_2 = list(range(second_class * 1000, (second_class + 1) * 1000))

idx_list = idx_list_1 + idx_list_2

In [6]:
w2_1_vs_2 = comparison_utils.slice_from_dist_matrix(pairwise_eud_dist_w2, idx_list=idx_list)
sw2_1_vs_2 = comparison_utils.slice_from_dist_matrix(pairwise_eud_dist_sw2, idx_list=idx_list)

labels = np.repeat([0, 1], [1000, 1000])

np.random.seed(42)
permutation = np.random.permutation(2000)

w2_1_vs_2 = w2_1_vs_2[permutation, :][:, permutation]
sw2_1_vs_2 = sw2_1_vs_2[permutation, :][:, permutation]

labels = labels[permutation]

train_length = 2000 // 5 * 4
test_length = 2000 // 5

w2_train_matrix = w2_1_vs_2[:train_length, :train_length]
sw2_train_matrix = sw2_1_vs_2[:train_length, :train_length]
print(w2_train_matrix.shape)
print(sw2_train_matrix.shape)

w2_test_matrix = w2_1_vs_2[train_length:, :train_length]
sw2_test_matrix = sw2_1_vs_2[train_length:, :train_length]
print(w2_test_matrix.shape)
print(sw2_test_matrix.shape)


train_labels = labels[:train_length]
test_labels = labels[train_length:]

neighbor_list = list(range(1, train_length // 2, 2))

w2_best_k = comparison_utils.kNNGridSearchCV(w2_train_matrix, train_labels, neighbor_list, nFolds=5)
sw2_best_k = comparison_utils.kNNGridSearchCV(sw2_train_matrix, train_labels, neighbor_list, nFolds=5)

w2_best_k = w2_best_k["n_neighbors"]
sw2_best_k = sw2_best_k["n_neighbors"]
print(w2_best_k)
print(sw2_best_k)

w2_metric_dict, w2_avg_auc, w2_std_auc = comparison_utils.kNNPerformanceEvaluation(
    w2_train_matrix, w2_test_matrix, train_labels, test_labels, w2_best_k
)
sw2_metric_dict, sw2_avg_auc, sw2_std_auc = comparison_utils.kNNPerformanceEvaluation(
    sw2_train_matrix, sw2_test_matrix, train_labels, test_labels, sw2_best_k
)


(1600, 1600)
(1600, 1600)
(400, 1600)
(400, 1600)


/opt/anaconda3/envs/MultispeciesOT/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


11
19


In [7]:
print(f"AUC of W2: {w2_avg_auc:.3f} +- {w2_std_auc:.3f}")
print(f"AUC of SW2: {sw2_avg_auc:.3f} +- {sw2_std_auc:.3f}")

AUC of W2: 0.889 +- 0.020
AUC of SW2: 0.755 +- 0.045


In [8]:
w2_all_vs_all = np.copy(pairwise_eud_dist_w2)
sw2_all_vs_all = np.copy(pairwise_eud_dist_sw2)

labels = np.repeat(
    [0, 1, 2, 3, 4, 5, 6, 7], [1000, 1000, 1000, 1000, 1000, 1000, 1000, 10000]
)

In [9]:
np.random.seed(42)
permutation = np.random.permutation(8000)

w2_all_vs_all = w2_all_vs_all[permutation, :][:, permutation]
sw2_all_vs_all = sw2_all_vs_all[permutation, :][:, permutation]

labels = labels[permutation]

In [10]:
train_length = 8000 // 5 * 4
test_length = 8000 // 5

w2_train_matrix = w2_all_vs_all[:train_length, :train_length]
sw2_train_matrix = sw2_all_vs_all[:train_length, :train_length]
print(w2_train_matrix.shape)
print(sw2_train_matrix.shape)

w2_test_matrix = w2_all_vs_all[train_length:, :train_length]
sw2_test_matrix = sw2_all_vs_all[train_length:, :train_length]
print(w2_test_matrix.shape)
print(sw2_test_matrix.shape)


train_labels = labels[:train_length]
test_labels = labels[train_length:]

neighbor_list = list(range(3, train_length//32, 4))

w2_best_k = comparison_utils.kNNGridSearchCV(w2_train_matrix, train_labels, neighbor_list, nFolds=5)
sw2_best_k = comparison_utils.kNNGridSearchCV(sw2_train_matrix, train_labels, neighbor_list, nFolds=5)

w2_best_k = w2_best_k["n_neighbors"]
sw2_best_k = sw2_best_k["n_neighbors"]
print(w2_best_k)
print(sw2_best_k)

w2_avg_auc, w2_std_auc = comparison_utils.kNNPerformanceEvaluationAcc(
    w2_train_matrix, w2_test_matrix, train_labels, test_labels, w2_best_k
)
sw2_avg_auc, sw2_std_auc = comparison_utils.kNNPerformanceEvaluationAcc(
    sw2_train_matrix, sw2_test_matrix, train_labels, test_labels, sw2_best_k
)

print(w2_avg_auc, sw2_avg_auc)

(6400, 6400)
(6400, 6400)
(1600, 6400)
(1600, 6400)
111
31
0.458125 0.378125


In [11]:
print(f"ACC of W2: {w2_avg_auc:.3f} +- {w2_std_auc:.3f}")
print(f"ACC of SW2: {sw2_avg_auc:.3f} +- {sw2_std_auc:.3f}")

ACC of W2: 0.458 +- 0.022
ACC of SW2: 0.378 +- 0.022
